# Module 01: Information Theory, Nonlinearity & Partial Information Decomposition (PID)

**Core Mathematical Framework:**
- **Self-Information & Shannon Entropy:** $h(x) = -\log_2 P(x)$, $H(X) = -\sum P(x)\log_2 P(x)$
- **Mutual Information:** $I(X; Y) = H(X) - H(X \mid Y) = D_{KL}(P(X, Y) \parallel P(X)P(Y))$
- **Landauer's Thermodynamic Bound:** $E_{\text{erasure}} \ge k_B T \ln 2 \approx 2.87 \times 10^{-21}\text{ J}$ at $T = 300\text{ K}$
- **Williams & Beer (2010) 4-Atom PID Lattice:** $I(X_1, X_2; Y) = R + U_1 + U_2 + S$ with Co-Information $II = R - S$

In [ ]:
import subprocess
import sys
from pathlib import Path

def _ensure_toolkit_on_path():
    cwd = Path.cwd().resolve()
    for cand in (cwd, cwd.parent, cwd.parent.parent):
        if (cand / "core").is_dir():
            if str(cand) not in sys.path:
                sys.path.insert(0, str(cand))
            return cand
    clone_dir = cwd / "The-Collective-Intelligence-Toolkit"
    if not clone_dir.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/poria-azadi96/The-Collective-Intelligence-Toolkit.git", str(clone_dir)],
            check=True,
        )
    if str(clone_dir) not in sys.path:
        sys.path.insert(0, str(clone_dir))
    return clone_dir

REPO_ROOT = _ensure_toolkit_on_path()
import matplotlib.pyplot as plt
import ipywidgets as widgets
from core.info_theory_pid import (
    create_pid_dashboard_figure,
    run_module01_tier1,
    run_module01_tier2,
    run_module01_tier3,
)
from core.real_world_demos import (
    demo_m01_computational_neuroscience,
    demo_m01_genomics_epistasis,
    demo_m01_thermodynamic_reversible_computing,
)

def run_pid_dashboard(
    gate_type='XOR',
    input_corr=0.0,
    noise_prob=None,
    sigma=0.0,
    coupling=1.0,
    temperature_k=300.0,
):
    fig, stats = create_pid_dashboard_figure(
        gate_type=gate_type,
        input_corr=input_corr,
        sigma=sigma,
        coupling=coupling,
        noise_prob=noise_prob,
        temperature_k=temperature_k,
    )
    plt.show()
    r = stats['pid']
    print(
        f"H(Y)={r['H(Y)']:.4f} b | Joint MI={r['I(X1, X2; Y)']:.4f} b | "
        f"R={r['Redundancy']:.4f} b | U1={r['Unique_X1']:.4f} b | "
        f"U2={r['Unique_X2']:.4f} b | Synergy={r['Synergy']:.4f} b | "
        f"Landauer={r['Landauer_J']:.3e} J ({r['Landauer_kTln2']:.2f} kT ln2)"
    )

widgets.interact(
    run_pid_dashboard,
    gate_type=widgets.Dropdown(
        options=['XOR', 'AND', 'OR', 'COPY', 'IDENTICAL', 'Continuous_Gaussian'],
        value='XOR',
        description='Mechanism:',
    ),
    input_corr=widgets.FloatSlider(min=-0.95, max=0.95, step=0.05, value=0.0, description='Corr (rho):'),
    noise_prob=widgets.fixed(None),
    sigma=widgets.FloatSlider(min=0.0, max=1.0, step=0.05, value=0.0, description='Noise (sigma):'),
    coupling=widgets.FloatSlider(min=0.0, max=1.0, step=0.05, value=1.0, description='Coupling (c):'),
    temperature_k=widgets.FloatSlider(min=4.0, max=400.0, step=10.0, value=300.0, description='Temp (K):'),
);

## 3-Tier Experimental Protocol
- **Tier 1 (Observation):** Verify `XOR` ($ho=0, \sigma=0.1, c=1.0$) exhibits $100\%$ Synergy ($S \approx 1.0\text{ bit}$) with $I(X_1;Y)\approx I(X_2;Y)\approx 0$, plus circular nonlinearity ($r \approx 0, I > 0$).
- **Tier 2 (Analysis):** Sweep $\rho \in [0, 0.95]$ on `AND` to trace the Redundancy–Synergy crossover and $II = R - S$ crossing zero.
- **Tier 3 (Stress Test):** Sweep $\sigma \in [0, 1.0]$ across all 4 gates to quantify entropy production, MI decay, and Landauer heat dissipation.

In [ ]:
def run_m01_tier_selector(tier='Tier 1: XOR Synergy & Canonical Gates'):
    if tier.startswith('Tier 1'):
        fig, res = run_module01_tier1(sigma=0.1, coupling=1.0)
        plt.show()
        print('XOR Stats:', res['gates']['XOR'])
        print('Circular Nonlinearity r =', round(res['nonlinearity']['pearson_r'], 4), '| KSG MI =', round(res['nonlinearity']['mi_ksg_bits'], 4), 'bits')
    elif tier.startswith('Tier 2'):
        fig, res = run_module01_tier2(gate_type='AND')
        plt.show()
        print('Crossover rho* where II = R - S = 0:', res['crossover_rho'])
    else:
        fig, _ = run_module01_tier3(temperature_k=300.0)
        plt.show()

widgets.interact(
    run_m01_tier_selector,
    tier=widgets.Dropdown(
        options=[
            'Tier 1: XOR Synergy & Canonical Gates',
            'Tier 2: AND Redundancy-Synergy Crossover',
            'Tier 3: Noise & Landauer Heat Stress Test',
        ],
        value='Tier 1: XOR Synergy & Canonical Gates',
        description='Protocol:',
    ),
);

## Module 01 Real-World Applications
1. **Computational Neuroscience & Brain Imaging:** Coincidence detection vs. redundant cortical firing.
2. **Genomics & Epistasis:** Discovering pure nonlinear SNP-SNP synergy where single-locus $r \approx 0$.
3. **Thermodynamic & Reversible Computing:** Irreversible CMOS reset vs. reversible Fredkin/Toffoli logic.

In [ ]:
def run_m01_real_world_demo(application='1. Computational Neuroscience'):
    if application.startswith('1.'):
        fig, res = demo_m01_computational_neuroscience()
    elif application.startswith('2.'):
        fig, res = demo_m01_genomics_epistasis()
    else:
        fig, res = demo_m01_thermodynamic_reversible_computing()
    plt.show()
    print(res)

widgets.interact(
    run_m01_real_world_demo,
    application=widgets.Dropdown(
        options=[
            '1. Computational Neuroscience',
            '2. Genomics & Epistasis',
            '3. Thermodynamic & Reversible Computing',
        ],
        value='1. Computational Neuroscience',
        description='Case Study:',
    ),
);